# ⚽ Match Predictor (H/D/A) - Premier League 2025-26
Este notebook entrena un modelo de **Regresión Logística** para predecir el resultado final de los partidos utilizando las estadísticas móviles (Rolling Stats) calculadas anteriormente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

%matplotlib inline

## 1. Carga de Datos Procesados

In [ ]:
df = pd.read_csv('../data/matches_processed.csv')
print(f"Dataset cargado con {df.shape[0]} partidos.")
df.head()

## 2. Definición de Features y Target

In [ ]:
predictors = [
    'rolling_goals_scored_h', 'rolling_goals_conceded_h', 'rolling_shots_h', 'rolling_shots_ot_h',
    'rolling_goals_scored_a', 'rolling_goals_conceded_a', 'rolling_shots_a', 'rolling_shots_ot_a'
]

X = df[predictors]
y = df['ftr'] # Target: H (Home), D (Draw), A (Away)

## 3. División de Entrenamiento y Prueba

In [ ]:
# Dividimos chronológicamente (usando una fecha de corte o simplemente shuffle=False si están ordenados)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Entrenamiento: {X_train.shape[0]} partidos")
print(f"Prueba: {X_test.shape[0]} partidos")

## 4. Entrenamiento del Modelo de Regresión Logística

In [ ]:
model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000)
model.fit(X_train, y_train)

## 5. Evaluación del Modelo

In [ ]:
y_pred = model.predict(X_test)
acc = accuracy_score(y_test, y_pred)

print(f"--- Accuracy del Modelo: {acc:.2%} ---")
print("\nReporte de Clasificación:")
print(classification_report(y_test, y_pred))

In [ ]:
# Matriz de Confusión
cm = confusion_matrix(y_test, y_pred, labels=['H', 'D', 'A'])
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['H', 'D', 'A'], yticklabels=['H', 'D', 'A'])
plt.title('Matriz de Confusión - Resultado del Partido')
plt.xlabel('Predicción')
plt.ylabel('Realidad')
plt.show()

## 6. Comparación contra Bet365 (Benchmark 49.8%)
Calculamos la precisión de las cuotas de apuestas solo para los partidos de prueba.

In [ ]:
# Identificar el resultado que la casa de apuestas considera más probable
test_indices = X_test.index
b365_results = []

for idx in test_indices:
    odds = [df.loc[idx, 'b365h'], df.loc[idx, 'b365d'], df.loc[idx, 'b365a']]
    best_odd = min(odds)
    if best_odd == odds[0]: b365_results.append('H')
    elif best_odd == odds[1]: b365_results.append('D')
    else: b365_results.append('A')

b365_acc = accuracy_score(y_test, b365_results)
print(f"Accuracy de Bet365 en este set: {b365_acc:.2%}")
print(f"Nuestra Diferencia: {acc - b365_acc:+.2%}")